In [ ]:
from langchain_ollama import ChatOllama
from langchain_ibm import ChatWatsonx
from langchain_core.prompts import (
    PromptTemplate,
    ChatPromptTemplate,
    MessagesPlaceholder,
)
from langchain_core.output_parsers import (
    StrOutputParser,
    JsonOutputParser,
    PydanticOutputParser,
)
from langchain_core.runnables import (
    RunnablePassthrough,
    RunnableParallel,
    RunnableLambda,
)
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.chat_history import (
    InMemoryChatMessageHistory,
    BaseChatMessageHistory,
)
from langchain_core.runnables.history import RunnableWithMessageHistory
from pydantic import BaseModel, Field
from typing import Literal
from dotenv import load_dotenv
import os
from langchain_community.document_loaders import (
    PyPDFLoader,
    CSVLoader,
    WebBaseLoader,
    DirectoryLoader,
)
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_ibm import WatsonxEmbeddings
from langchain_chroma import Chroma
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI

from langchain_classic.chains.query_constructor.base import AttributeInfo
from langchain_classic.retrievers import (
    EnsembleRetriever,
    ContextualCompressionRetriever,
    BM25Retriever,
)
from langchain_classic.retrievers.self_query.chroma import ChromaTranslator
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

from langchain_cohere import CohereRerank
from langchain_classic.retrievers.document_compressors import LLMChainExtractor, EmbeddingsFilter, DocumentCompressorPipeline

In [ ]:
# .env 내용 가져오기
load_dotenv()

apikey = os.getenv("WATSONX_API_KEY")
project_id = os.getenv("WATSONX_PROJECT_ID")
watsonx_ai_url = os.getenv("WATSONX_URL")
hf_token = os.getenv("HF_TOKEN")
COHERE_API_KEY = os.getenv("COHERE_API_KEY")

watson_llm = ChatWatsonx(
    model_id="ibm/granite-4-h-small",
    url=f"{watsonx_ai_url}",
    api_key=f"{apikey}",
    project_id=f"{project_id}",
    max_tokens=2000,
    temperature=0
)

ollama_embedding = OllamaEmbeddings(model="nomic-embed-text-v2-moe")
watson_embedding = WatsonxEmbeddings(
    model_id="ibm/granite-embedding-278m-multilingual",
    url=f"{watsonx_ai_url}",
    api_key=f"{apikey}",
    project_id=f"{project_id}",
    params = {
        "temperature" : 0
    }
)

qwen_llm = ChatOllama(model="qwen3.5:4b", temperature=0)
exaone_llm = ChatOllama(model="exaone3.5:2.4b", temperature=0)

### RAG 심화




In [ ]:
# pdf => chunks 반환 함수

def create_chunks_from_pdf(pdf_path, chunk_size=500, chunk_overlap=50):
    loader = PyPDFLoader(pdf_path)
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunks = splitter.split_documents(loader.load())

    chunks = [chunk for chunk in chunks if chunk.page_content.strip()]

    return chunks

def create_vectorstore(chunks, embeddings, collection_name, persis_directory='./db/chroma_db'):
    db = Chroma(
        embedding_function=embeddings,
        persist_directory=persis_directory,
        collection_name=collection_name,
    )
    
    db.delete_collection()

    return Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=persis_directory,
        collection_name=collection_name,
    )


def create_retriever(
    vectorstore, search_type="similarity", k=3, fetch_k=20, lambda_mult=0.5
):
    kwargs = {"k":k}

    if search_type=="mmr":
        kwargs['fetch_k'] = fetch_k
        kwargs['lambda_mult'] = lambda_mult

    return vectorstore.as_retriever(search_type=search_type, search_kwargs=kwargs)


def print_retrieved_docs(title, retriever, query):
    docs = retriever.invoke(query)

    print("\n" + "="*50)
    print(title)
    print("\n" + "="*50)    

    for i, doc in enumerate(docs):
        print(f"\n [chunk {i}]")
        print(doc.page_content)
        print(f"\nPage : {doc.metadata.get("page")}")

In [ ]:
chunks1 = create_chunks_from_pdf("./data/Summary of ChatGPTGPT-4 Research.pdf", 1000, 100)
chunks2 = create_chunks_from_pdf("./data/Summary of ChatGPTGPT-4 Research.pdf", 300, 30)

print(f"분할된 청크 수 : {len(chunks1)}")
print(f"분할된 청크 수 : {len(chunks2)}")

In [ ]:
vectorstore1 = create_vectorstore(chunks1, ollama_embedding, collection_name="gpt_research_ollama3")
vectorstore2 = create_vectorstore(chunks1, ollama_embedding, collection_name="gpt_research_ollama4")

ollama1_retriever = create_retriever(vectorstore1)
ollama2_retriever = create_retriever(vectorstore2)

query = 'where can i use chatGPT?'

print_retrieved_docs('ollama embedding', ollama1_retriever, query)
print_retrieved_docs('ollama embedding', ollama2_retriever, query)

### 2. MMR (Maximal Marginal Relevance) Retriever
 - 관련성(Relevace)과 다양성(Diversity) 고려
 - 법률 문서, 기술 매뉴얼 유사 내용이 반복되는 문서에서 효과적이다.

In [ ]:
chunks1 = create_chunks_from_pdf(
    "./data/2026 상 삼성전자 DX부문 직무기술서.pdf",
    1000,
    100,
)

vectorstore1 = create_vectorstore(
    chunks1, ollama_embedding, collection_name="samsung_watson1", persist_directory="./db/samsung_watson_db"
)

mmr_retriever = create_retriever(vectorstore1, search_type="mmr", k=5)
similarity_retriever = create_retriever(vectorstore1, k=5)

query = '마케팅 - 제품/서비스 마케팅 포지션은?'

print_retrieved_docs('MMR', mmr_retriever, query)
print_retrieved_docs('similarity', similarity_retriever, query)


# 3. selfquery retriever
- 자연어 질문을 분석하여 시멘틱 검색쿼리와 메타데티어 필터를 llm이 자동 생성하게 하는 고급 retriever
- 질문 : 2023 년 이후 기약금액이 1억 이상인 계약 찾아줘 => LLM
    시멘틱 검색 쿼리 : 계약
    filter : {year >= 2023, 계약금액 >= 100000}

In [ ]:
docs = [
    Document(
        page_content="삼성전자 제품 마케팅 직무입니다.",
        metadata={
            "year":2025,
            "department":"marketing"
        }
    ),
        Document(
        page_content="삼성전자 회계 직무 입니다.",
        metadata={
            "year":2024,
            "department":"accounting"
        }
    ),
        Document(
        page_content="삼성전자 백엔드 직무입니다.",
        metadata={
            "year":2023,
            "department":"backend"
        }
    )
]

metadata_field_info = [
    AttributeInfo(
        name="year", description="문서작성 연도", type="integer"),
    AttributeInfo(
        name="department", description="직무 부서", type="string"
    )
]


document_content_description = "회사 내부 문서 및 직무 자료"

In [ ]:
vectorstore1 = create_vectorstore(
    docs,
    watson_embedding,
    collection_name="selfquery",
    persis_directory='./db/watson_chroma'
)

self_query_retriever = SelfQueryRetriever.from_llm(
    llm=watson_llm,
    vectorstore=vectorstore1,
    metadata_field_info=metadata_field_info,
    document_contents=document_content_description,
    verbose=True,
    enable_limit=True,
    structured_query_translator=ChromaTranslator()
)


In [ ]:
question = "2023년 이후 backend 부서 직무를 찾아줘"
self_query_retriever.invoke(question)

In [ ]:
docs = [
    Document(
        page_content="수분 가득한 히알루론산 세럼으로 피부 속 깊은 곳 까지 수분을 공급합니다.",
        metadata={"year": 2024, "category": "스킨케어", "user_rating": 4},
    ),
    Document(
        page_content="24시간 지속되는 매트한 피니시의 파운데이션, 모공을 커버하고 자연스러운 피부 표현이 가능합니다.",
        metadata={"year": 2023, "category": "메이크업", "user_rating": 3},
    ),
    Document(
        page_content="비타민 C 함유 브라이트닝 크림, 칙칙한 피부톤을 환하게 밝혀줍니다.",
        metadata={"year": 2023, "category": "스킨케어", "user_rating": 4},
    ),
    Document(
        page_content="자외선 차단 기능이 있는 톤업 선크림, SPF50+/PA+++ 높은 자외선 차단지수로 피부를 보호합니다.",
        metadata={"year": 2025, "category": "선케어", "user_rating": 5},
    ),
    Document(
        page_content="롱래스팅 립스틱, 선명한 발색과 촉촉한 사용감으로 하루종일 편안하게 사용 가능합니다.",
        metadata={"year": 2024, "category": "메이크업", "user_rating": 4},
    ),
]

# 메타데이터 필드 정보 생성
metadata_field_info = [
    AttributeInfo(name="year", description="화장품 출시 연도", type="integer"),
    AttributeInfo(
        name="category",
        description="화장품 카테고리 ['스킨케어', '메이크업', '클렌징', '선케어']",
        type="string",
    ),
    AttributeInfo(
        name="user_rating",
        description="화장품 평점 (1~5)",
        type="integer",
    ),
]

document_content_description = "화장품 제품 정보"

vectorstore1= create_vectorstore(
    docs,
    watson_embedding,
    collection_name="selfquery2",
    persis_directory="./db/watson_chroma"
)

self_query_retriever = SelfQueryRetriever.from_llm(
    llm=watson_llm,
    vectorstore=vectorstore1,
    metadata_field_info=metadata_field_info,
    document_contents=document_content_description,
    verbose=True,
    enable_limit=True,
    structured_query_translator=ChromaTranslator(),
)

In [ ]:
self_query_retriever.invoke("2024년 이후로 평점이 4 이상인 제품을 추천해줘")

In [ ]:
self_query_retriever.invoke("카테고리가 선케어인 상품을 추천해줘")

### OCR

In [ ]:
pdf_path = "./data/2026 상 삼성전자 DX부문 직무기술서.pdf"
chunks = create_chunks_from_pdf(pdf_path)
for i in range(2):
    print("="*50)
    print(chunks[i].page_content[:500])

In [ ]:
# PDF를 이미지로 변환하기
import os

import fitz

output_path = "./output"
os.makedirs(output_path, exist_ok=True)

pdf = fitz.open(pdf_path)

for page_num in range(len(pdf)):
    page = pdf[page_num]
    pix = page.get_pixmap(dpi=300)
    pix.save(os.path.join(output_path, f"page_{page_num}.png"))

pdf.close()

In [ ]:
import easyocr

reader = easyocr.Reader(["ko", "en"])
result = reader.readtext("./output/page_19.png", detail=0, paragraph=True)
print("\n".join(result))

In [ ]:
import json

pages = []

for page_num in range(31):
    image_path = f"./output/page_{page_num}.png"
    result = reader.readtext(image_path, detail=0, paragraph=True)
    text = "\n".join(result)
    pages.append({"page": page_num+1, "content": text})

with open("./data/samsung_dx_ocr.json", "w", encoding="utf-8") as f:
    json.dump(pages, f, ensure_ascii=False, indent=2)

### BM25

- 유사도 검색은 유사성 위주 (정확한 제품명, 코드, 버전, 고유명사 등의 키워드 매칭에는 약함)
- 유사도 검색의 단점 보완 (semantic + sparse)
- 예
    - 환불 가능한 기간이 어떻게 되나요? 유사도 검색 유리
    - ERR_CONNECTION 오류 -> 키워드 검색

In [ ]:
with open("./data/samsung_dx_ocr.json", "r", encoding="utf-8") as f:
    pages = json.load(f)

docs = []

for page in pages:
    docs.append(
        Document(
            page_content=page["content"],
            metadata={
                "page": page["page"],
                "source": "samsung_dx_ocr",
                
                },
        )
    )

In [ ]:
# len (docs)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

chunks = splitter.split_documents(docs)

vectorstore = create_vectorstore(
    chunks=chunks,
    embeddings=watson_embedding,
    persis_directory="./db/chroma_db",
    collection_name="samsung_dx"
)

In [ ]:
# 유사도 검색
dense_retriever = create_retriever(vectorstore)

# 키워드 검색(법령, 제품명...)
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 4

hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, bm25_retriever],
    weights=[0.5, 0.5]
)

query = "시스템 소프트웨어 자격 요건에 운영체제 개념도 포함되어 있는가?"

print_retrieved_docs("hybrid", hybrid_retriever, query)

### ReRanking & Contextual Compression
- 유사도 검색 (Bi-Encoder) : 쿼리(질문)와 문서를 각각 벡터로 변환 후 계산
- Cross-Encoder : 쿼리+문서 쌍으로 벡터로 변환
    - 정확도가 매우 높으나, 매우 느림

In [ ]:
!pip install cohere langchain-cohere sentence-transformers

In [ ]:
# 기본검색(유사도 검색)
base_retriever = dense_retriever = create_retriever(vectorstore, k=20)
query = "서울 근무 직무는 어떤 항목에 지원해야하나요?"
docs = base_retriever.invoke(query)

# 20개 후보군을 대상으로 reranking
reranker = CohereRerank(model="rerank-v4.0-pro", top_n=5)

compression_retriever = ContextualCompressionRetriever(base_compressor=reranker, base_retriever=base_retriever)

# 최종
print_retrieved_docs("Rerank", compression_retriever, query)

### LLMChainExtractor

- 질문+문서를 같이 LLM에게 보내서 질문과 관련된 내용만 추출
- 현재 chunks 안에 있는 내용을 줄여내는 것

In [ ]:
# 문서로드 / 청크 추출
chunks = create_chunks_from_pdf("./data/제주관광가이드.pdf")

# 인덱싱 - vectorstore
vectorstore = create_vectorstore(chunks, watson_embedding, collection_name="jeju_guide")

# 질의
base_retriever = create_retriever(vectorstore=vectorstore, k=20)

docs = base_retriever.invoke("생활 속 제주어 에서 엄불랑하다는 무슨 뜻이야?")

for doc in docs:
    print(doc.page_content[:500])
    print('-'*10)

In [ ]:
# 답변만 축약

extractor = LLMChainExtractor.from_llm(watson_llm)

compression_retriever = ContextualCompressionRetriever(base_compressor=extractor, base_retriever=base_retriever)

docs = compression_retriever.invoke("안녕하세요를 제주어로 하면 무엇인가?")

for doc in docs:
    print(doc.page_content[:500])
    print('-'*10)

### Embedding Filter

- 임계값을 기준으로 미달한 문서 제외

In [ ]:
embedding_filter = EmbeddingsFilter(embeddings=watson_embedding, similarity_threshold=0.8)

pipeline = DocumentCompressorPipeline(transformers=[embedding_filter, extractor])

compression_retriever = ContextualCompressionRetriever(base_compressor=pipeline, base_retriever=base_retriever)

docs = compression_retriever.invoke("참말로 좋습니다를 제주어로 한다면?")

for doc in docs:
    print(doc.page_content[:500])
    print('-'*10)

In [ ]:
filtered_docs = embedding_filter.compress_documents(docs,query)

len(filtered_docs)

### RAG 성능 개선
1. 문서 전처리
- chunk 최적화, metadata 추가

2. Retrieval(검색) 개선
- MMR, BM25, Hybrid, SelfQuery 이용

3. Retrieval 후 처리
- Rerank, EmbeddingFilter, LLMChainExtractor, ContextualCompressionRetriever

In [ ]:
import easyocr

reader = easyocr.Reader(["ko", "en"])

input_pdf = "./data/한양대 대학원 캠퍼스 가이드.pdf"
path_name = "hanyang_graduate"


def convert_pdf_to_image(path_name, pdf_path):
    import os
    import fitz

    output_path = f"./output/{path_name}"
    os.makedirs(output_path, exist_ok=True)

    pdf = fitz.open(pdf_path)

    for page_num in range(len(pdf)):
        page = pdf[page_num]
        pix = page.get_pixmap(dpi=300)
        pix.save(os.path.join(output_path, f"page_{page_num}.png"))

    pdf.close()


def convert_image_to_json(path_name):
    import os
    import json

    image_dir = f"./output/{path_name}"
    num_pages = len([f for f in os.listdir(image_dir) if f.endswith(".png")])

    pages = []

    for page_num in range(num_pages):
        image_path = f"{image_dir}/page_{page_num}.png"
        result = reader.readtext(image_path, detail=0, paragraph=True)
        text = "\n".join(result)
        pages.append({"page": page_num + 1, "content": text})

    os.makedirs(f"./data/{path_name}", exist_ok=True)
    with open(f"./data/{path_name}/result.json", "w", encoding="utf-8") as f:
        json.dump(pages, f, ensure_ascii=False, indent=2)


# 이미지 기반 PDF라 PyPDFLoader로는 텍스트가 안 나옴.
# OCR 결과(result.json)를 읽어서 chunk를 만든다.
def create_chunks_from_json(path_name, chunk_size=500, chunk_overlap=50):
    import json
    from langchain.schema import Document

    with open(f"./data/{path_name}/result.json", encoding="utf-8") as f:
        pages = json.load(f)

    docs = [
        Document(page_content=p["content"], metadata={"page": p["page"]})
        for p in pages
        if p["content"].strip()
    ]

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap
    )
    chunks = splitter.split_documents(docs)
    chunks = [chunk for chunk in chunks if chunk.page_content.strip()]
    print(len(chunks))

    return chunks


def create_vectorstore(
    chunks, embeddings, collection_name, persist_directory
):

    db = Chroma(
        embedding_function=embeddings,
        persist_directory=persist_directory,
        collection_name=collection_name,
    )

    db.delete_collection()

    return Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=persist_directory,
        collection_name=collection_name,
    )


# 1. PDF -> 이미지 -> OCR(json)
convert_pdf_to_image(path_name=path_name, pdf_path=input_pdf)
convert_image_to_json(path_name=path_name)

# 2. OCR 결과(json) -> chunk -> 벡터스토어
chunks = create_chunks_from_json(path_name)
create_vectorstore(
    chunks=chunks,
    embeddings=watson_embedding,
    collection_name=path_name,
    persist_directory=f"./db/{path_name}",
)

In [ ]:
hanyang_vectorstore = \
    Chroma(
        embedding_function=watson_embedding,
        persist_directory=f"./db/{path_name}",
        collection_name=path_name,
    )

### 소비자보호원 서비스집단 분쟁조정 사례집 RAG

In [ ]:
from langchain_ibm import ChatWatsonx
from langchain_core.prompts import (
    PromptTemplate,
    ChatPromptTemplate,
    MessagesPlaceholder,
)

from langchain_core.output_parsers import (
    StrOutputParser,
    JsonOutputParser,
    PydanticOutputParser,
)

from dotenv import load_dotenv
import os

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_ibm import WatsonxEmbeddings
from langchain_chroma import Chroma

from pydantic import BaseModel, Field
from langchain_core.runnables import RunnablePassthrough

import re

from pprint import pprint

load_dotenv()

apikey=  os.getenv("WATSONX_API_KEY")
project_id = os.getenv("WATSONX_PROJECT_ID")
watson_ai_url = os.getenv("WATSONX_URL")

watson_llm = ChatWatsonx(
    apikey=apikey,
    project_id=project_id,
    watsonx_url=watson_ai_url,
    model_id="ibm/granite-4-h-small",
    max_tokens=2000,
    params = {
        "temperature": 0
    }
)

watson_embedding = WatsonxEmbeddings(
    model_id="ibm/granite-embedding-278m-multilingual",
    url=f"{watson_ai_url}",
    api_key=f"{apikey}",
    project_id=f"{project_id}",
    params={"temperature": 0},
)

# 문서 로드
pdf_path = "./data/2018 서비스·집단 분쟁조정 사례집.pdf"

loader = PyPDFLoader(pdf_path)
docs = loader.load()
print(len(docs)) # 예상 값 : 200
pprint(docs[0].page_content)
pprint(docs[0].metadata)

# 전처리 - 정규식을 이용한 내용 추출
pattern = r"사\s*례\s*\d+.*?사건번호.*?결정일자.*?\d{4}\.\s?\d{1,2}\.\s?\d{1,2}\."

target_text = "".join(docs[10].page_content)
split_text = re.findall(pattern, target_text, re.DOTALL)

parts = []
if split_text: # 패턴이 존재할때 패턴을 기준으로 문서 분리
    parts = re.split(pattern, target_text)

    # 사례번호
    case_no = re.findall(r"례\s?(\d+)\s?사건번호", split_text[0])
    print(case_no[0] if case_no else "사례번호 미검출")
else:
    print("패턴 매칭 결과 없음")

# 사건 번호, 결정일자 추출
class CaseMetadata(BaseModel):
    case_number:str = Field(description="사건번호 예: 2018일나565")
    decision_date:str = Field(description="결정일자 예: 2018. 8. 7")


metadata_prompt = PromptTemplate.from_template(
    """\
        다음은 분쟁 조정 사례에 대한 텍스트입니다.

        - case_number : 사건번호
        - decision_date : 결정일자

        반드시 JSON 으로만 반환하세요
        {case_text}
    """
)

structed_llm = watson_llm.with_structured_output(CaseMetadata)
chain = metadata_prompt | structed_llm

if split_text:
    case_metadata = chain.invoke({
        "case_text" : split_text[0]
    })

    print(case_metadata)
    print(dict(case_metadata))


# 주 문 ~~~ (탐색용: parts 가 분할됐고 두 번째 조각이 있을 때만)
if len(parts) > 1:
    pprint(parts[1])

    m = re.search(r"주 문\n", parts[1])
    if m:
        title = parts[1][:m.span()[0]].replace("\n", "").strip()      # 주문 앞부분 = 제목
        content = parts[1][m.span()[0]:].strip()                       # 주문 이후 = 내용
        pprint(title)
        pprint(content)

# 사례 번호, 사건 번호, 결정일자, 제목은 meta데이터 추가
docs = []
case_metadata = {}

# 사건이 시작되는 페이지 ~ 마지막에서 -2 페이지 까지 반복
for doc in docs[10:-2]:
    split_text = re.findall(pattern, "".join(doc.page_content), re.DOTALL)
    if split_text:  # 새 사례가 시작되는 페이지
        case_metadata = {}

        # 사례 번호 추출
        case_no = re.findall(r"례\s?(\d+)\s?사건번호", split_text[0])
        case_metadata["case_no"] = case_no[0] if case_no else ""

        # 패턴 기준으로 텍스트 분할
        parts = re.split(pattern, "".join(doc.page_content))

        if len(parts) > 1 and re.search(r"주 문\n", parts[1]):
            span = re.search(r"주 문\n", parts[1]).span()
            # 제목 추출 (주문 앞부분)
            case_metadata["title"] = parts[1][:span[0]].replace("\n", "").strip()
            # 내용 추출 후 기존 내용 업데이트 (주문 이후)
            doc.page_content = parts[1][span[0]:].strip()
        else:
            case_metadata["title"] = ""

        # 사건 번호, 결정 일자 추출
        i = 0
        while i < 10:
            try:
                response = chain.invoke({"case_text": split_text[0]})
                for k, v in dict(response).items():
                    case_metadata[k] = v.replace("\n", "").replace(" ", "")
                break
            except Exception:
                i += 1
                continue

        doc.metadata.update(case_metadata)
        docs.append(doc)
    else:
        doc.metadata.update(case_metadata)
        docs.append(doc)

len(docs)

# 2. 분할

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

split_docs = splitter.split_documents(docs)

print(split_docs[0].page_content)

In [ ]:
# 마침표 뒤에 나오는 줄바꿈 문자를 그대로 두고 나머지 줄바꿈 문자만 제거
pprint(split_docs[18].page_content)

text = split_docs[18].page_content

text = re.sub(r"(?<!\.)\n"," ", text)
pprint(text)